# 第12回：通常のMCMCとレプリカ交換を比較する

演習20分・採点なし。CPUで上から実行します。第10回と同じ無次元模型 $U(x)=(x^2-1)^2+0.25x$ を使います。今回の目標は $k_BT=0.25$。通常法は同温度の4本、交換法は0.25、0.5、1、2の4本を同じステップ数だけ走らせます。エネルギー評価数を揃えますが、交換判定の仕事は追加されます。

In [ ]:
%pip -q install numpy matplotlib

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import json, time
OUT = Path("results"); OUT.mkdir(exist_ok=True)
def U(x): return (x*x-1)**2 + 0.25*x


## 1. 二種類の更新を読む

各温度でMetropolis更新を行い、10ステップごとに隣の温度と交換を試みます。

交換の対数比は $(β_i-β_j)(U_i-U_j)$。交換時は**座標とエネルギーと追跡ラベルを同時に交換**します。配列の列は温度スロットです。

In [ ]:
def run(seed, exchange, steps=20000, sigma=0.15):
    rng=np.random.default_rng(seed)
    temperatures=np.array([0.25,0.5,1.,2.]) if exchange else np.full(4,0.25)
    x=np.full(4,-1.); energy=U(x); labels=np.arange(4)
    trajectory=np.empty((steps,4)); history=np.empty((steps,4),int)
    swaps=np.zeros(3); attempts=np.zeros(3)
    for k in range(steps):
        proposal=x+rng.normal(0,sigma,4); new_energy=U(proposal)
        accept=np.log(rng.random(4)) < np.minimum(0,-(new_energy-energy)/temperatures)
        x[accept]=proposal[accept];energy[accept]=new_energy[accept]
        if exchange and k%10==0:
            for i in range((k//10)%2,3,2):
                j=i+1;attempts[i]+=1
                logratio=(1/temperatures[i]-1/temperatures[j])*(energy[i]-energy[j])
                if np.log(rng.random()) < min(0,logratio):
                    x[[i,j]]=x[[j,i]];energy[[i,j]]=energy[[j,i]];labels[[i,j]]=labels[[j,i]];swaps[i]+=1
        trajectory[k]=x;history[k]=labels
    return trajectory,history,swaps/np.maximum(attempts,1)


In [ ]:
STEPS=20000;SEED=10
ordinary,_,_=run(SEED,False,STEPS)
replica,labels,swap_acceptance=run(SEED,True,STEPS)
burn=STEPS//5
normal_samples=ordinary[burn:].ravel()
exchange_samples=replica[burn:,0]
grid=np.linspace(-3,3,30001);density=np.exp(-U(grid)/0.25);density/=np.trapezoid(density,grid)
reference=np.trapezoid(density[grid>0],grid[grid>0])
print('各方法のエネルギー評価数:',4*STEPS)
print('理論 P(x>0):',reference)
print('通常4本 / 交換の目標温度:',np.mean(normal_samples>0),np.mean(exchange_samples>0))
print('隣接対の交換受理率:',swap_acceptance)

## 2. 温度別に標本を集める

交換法は**第0列だけ**を目標温度の標本に使います。4温度の列をそのまま混ぜると別の分布になります。通常法の4本は同温度なのでまとめられます。

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(12,4))
for ax,sample,title in zip(axes,[normal_samples,exchange_samples],['4 ordinary chains','Replica exchange: target slot']):
    ax.hist(sample,bins=65,density=True,alpha=.5);ax.plot(grid,density,color='black');ax.set(xlim=(-1.8,1.8),xlabel='x',ylabel='density',title=title)
fig.tight_layout();fig.savefig(OUT/'densities.png',dpi=160);plt.show()

## 3. 同じコピーが温度間を移動する履歴を見る

色線は一つのコピーの温度スロットです。交換による飛び移りを、実時間の分子の運動や反応速度と解釈しません。

In [ ]:
fig,axes=plt.subplots(2,1,figsize=(12,6))
axes[0].plot(replica[:4000,0],lw=.8);axes[0].set(ylabel='x at target slot')
for label in range(4): axes[1].plot(np.argmax(labels[:4000]==label,axis=1),lw=.6,label=f'copy {label}')
axes[1].set(xlabel='MC step (not physical time)',ylabel='temperature slot');axes[1].legend(ncol=4)
fig.tight_layout();fig.savefig(OUT/'exchange_trace.png',dpi=160);plt.show()

## 4. 乱数を変えて比較する

一度よく合っただけで「常に速い」と判断しません。seedごとの右の谷の割合を比べます。必要ならSTEPSを増やして再実行します。アンブレラは今回の演習範囲に含めません。

In [ ]:
rows=[]
for seed in [10,20,30,40,50]:
    a,_,_=run(seed,False,STEPS);b,_,_=run(seed,True,STEPS)
    rows.append([seed,float(np.mean(a[burn:]>0)),float(np.mean(b[burn:,0]>0))])
print('seed, ordinary, exchange');print(np.array(rows))
report={'steps':STEPS,'energy_evaluations_each':4*STEPS,'reference_right':float(reference),'results':rows,'interpretation':'有限標本の比較。交換履歴は物理時間ではない。'}
(OUT/'comparison.json').write_text(json.dumps(report,indent=2,ensure_ascii=False))
np.savez(OUT/'samples.npz',ordinary=ordinary,replica=replica,labels=labels)
import shutil
shutil.make_archive('bioinfo12_results','zip',OUT)
print('bioinfo12_results.zip を保存しました')